##### Copyright 2026 Google LLC.

In [2]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemini API: Enum Quickstart

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Enum.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

The Gemini API allows you to supply a schema to define function arguments (for [function calling](../quickstarts/Function_calling.ipynb)), or to constrain its output in [JSON](../quickstarts/JSON_mode.ipynb) or using an Enum. This tutorial gives some examples using enums.

> **Note:** This notebook uses the [Interactions API](https://ai.google.dev/gemini-api/docs/interactions), the latest way to interact with Gemini models. Looking for the `generateContent` version? Check the [archive branch](https://github.com/google-gemini/cookbook/blob/archive/generate-content-api/quickstarts/Enum.ipynb).

### Install dependencies

In [ ]:
%pip install -U -q "google-genai>=2.9.0"  # 2.9.0 for Interactions API

Note: you may need to restart the kernel to use updated packages.


### Configure your API key

To run the following cell, your API key must be stored in a Colab Secret named `GEMINI_API_KEY`. If you don't already have an API key, or you're not sure how to create a Colab Secret, see [Authentication](https://github.com/google-gemini/cookbook/blob/main/quickstarts/Authentication.ipynb) for a walkthrough.

In [9]:
from google import genai
from google.colab import userdata

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=GEMINI_API_KEY)

Select the model you want to use in this guide:

In [11]:
MODEL_ID = "gemini-3.6-flash" # @param ["gemini-3.5-flash-lite", "gemini-2.5-flash", "gemini-3.6-flash", "gemini-2.5-pro", "gemini-2.5-flash-preview", "gemini-3-flash-preview", "gemini-3.1-pro-preview"] {"allow-input":true, isTemplate: true}

## Enums

In the simplest case is you need the model to choose one option from a list of choices, use an enum class to define the schema. Ask it to identify this instrument:

In [14]:
!curl -O https://storage.googleapis.com/generativeai-downloads/images/instrument.jpg

import base64
from IPython.display import Image, display

display(Image(filename='instrument.jpg'))

with open('instrument.jpg', 'rb') as f:
    instrument_data = base64.b64encode(f.read()).decode('utf-8')

<IPython.core.display.Image object>


The response should be one of the following options:

In [16]:
import enum

class InstrumentClass(enum.Enum):
    PERCUSSION = "Percussion"
    STRING = "String"
    WOODWIND = "Woodwind"
    BRASS = "Brass"
    KEYBOARD = "Keyboard"

In the Interactions API, pass the enum class as the `schema` inside `response_format` with `application/json` mime type:


In [18]:
interaction = client.interactions.create(
    model=MODEL_ID,
    input=[
        {"type": "image", "data": instrument_data, "mime_type": "image/jpeg"},
        {"type": "text", "text": "what is the category of this instrument?"},
    ],
    response_format={
        "type": "text",
        "mime_type": "application/json",
        "schema": {
            "type": "string",
            "enum": [e.value for e in InstrumentClass],
        },
    },
)

print(interaction.steps[-1].content[0].text)


"Keyboard"


You can also use enums with `application/json` mime_type. In this simple case the response will be wrapped in a JSON string:

In [20]:
interaction = client.interactions.create(
    model=MODEL_ID,
    input=[
        {"type": "image", "data": instrument_data, "mime_type": "image/jpeg"},
        {"type": "text", "text": "what category of instrument is this?"},
    ],
    response_format={
        "type": "text",
        "mime_type": "application/json",
        "schema": {
            "type": "string",
            "enum": [e.value for e in InstrumentClass],
        },
    },
)

print(interaction.steps[-1].content[0].text)

"Keyboard"


Outside of simple multiple choice problems, an enum can be used anywhere in the schema for [JSON](../quickstarts/JSON_mode.ipynb) or [function calling](../quickstarts/Function_calling.ipynb). For example, ask it for a list of recipe titles, and use a `Grade` enum to give each one a popularity-grade:

In [22]:
import typing_extensions as typing

class Grade(enum.Enum):
    A_PLUS = 'a+'
    A = 'a'
    B = 'b'
    C = 'c'
    D = 'd'
    F = 'f'

class Recipe(typing.TypedDict):
    recipe_name: str
    grade: Grade

For this example you want a list of `Recipe` objects, so pass `list[Recipe]` to the `schema` in `response_format`:

In [24]:
result = client.interactions.create(
    model=MODEL_ID,
    input="List about 10 cookie recipes, grade them based on popularity",
    response_format={
        "type": "text",
        "mime_type": "application/json",
        "schema": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "recipe_name": {"type": "string"},
                    "grade": {
                        "type": "string",
                        "enum": [e.value for e in Grade],
                    },
                },
                "required": ["recipe_name", "grade"],
            },
        },
    },
)

In [25]:
import json
response = json.loads(result.steps[-1].content[0].text)
print(json.dumps(response, indent=4))

[
    {
        "recipe_name": "Chocolate Chip Cookies",
        "grade": "a+"
    },
    {
        "recipe_name": "Sugar Cookies",
        "grade": "a"
    },
    {
        "recipe_name": "Peanut Butter Cookies",
        "grade": "a"
    },
    {
        "recipe_name": "Snickerdoodles",
        "grade": "a"
    },
    {
        "recipe_name": "Oatmeal Raisin Cookies",
        "grade": "b"
    },
    {
        "recipe_name": "Shortbread Cookies",
        "grade": "b"
    },
    {
        "recipe_name": "French Macarons",
        "grade": "b"
    },
    {
        "recipe_name": "Gingerbread Cookies",
        "grade": "b"
    },
    {
        "recipe_name": "Fortune Cookies",
        "grade": "c"
    },
    {
        "recipe_name": "Biscotti",
        "grade": "c"
    }
]


## Next Steps
### Useful API references:

Check the [structured output](https://ai.google.dev/gemini-api/docs/structured-output) documentation or the [`GenerationConfig`](https://ai.google.dev/api/generate-content#generationconfig) API reference for more details.

### Related examples

* The constrained output is used in the [Text summarization](../examples/json_capabilities/Text_Summarization.ipynb) example to provide the model a format to summarize a story (genre, characters, etc...)
* The [Object detection](../examples/Object_detection.ipynb) examples are using the JSON constrained output to uniformize the output of the detection.

### Continue your discovery of the Gemini API

An Enum is not the only way to constrain the output of the model, you can also use an [JSON](../quickstarts/JSON_mode.ipynb) schema. [Function calling](../quickstarts/Function_calling.ipynb) and [Code execution](../quickstarts/Code_Execution.ipynb) are other ways to enhance your model by either let him use your own functions or by letting it write and run them.